# Fake News Detection: Model Training and Evaluation
### A Comparative Analysis of Four Machine Learning Classifiers

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import re
import string
import joblib

## 1. Importing Libraries
This cell imports all the necessary Python libraries for the project, including pandas for data manipulation, Scikit-learn for machine learning tasks, and others for text processing.

In [2]:
fake = pd.read_csv('Fake.csv')
true = pd.read_csv('True.csv')

## 2. Loading the Datasets
Here, we load the two separate datasets: `Fake.csv` containing fake news articles and `True.csv` containing real news articles. Each is loaded into its own pandas DataFrame.

In [3]:
fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [4]:
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


### 2.1. Exploring the Data
Let's preview the first few rows of each dataset to understand their structure and content.

## 3. Data Preparation and Cleaning

### 3.1. Labeling the Data
To prepare for supervised learning, we add a 'class' column to each DataFrame. We will use `0` to represent 'fake' news and `1` to represent 'real' news.

In [5]:
fake['class']=0
true['class']=1

In [6]:
fake.head()

,title,text,subject,date,class
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0


In [7]:
true.head()

,title,text,subject,date,class
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1


### 3.2. Merging the Datasets
The two separate DataFrames are concatenated into a single, comprehensive dataset. We then shuffle and display a random sample to ensure both classes are present.

In [8]:
data = pd.concat([fake, true], axis = 0)

In [9]:
data.sample(10)

,title,text,subject,date,class
1981,Sean Spicer Just Said Something About Trumpca...,"When Donald Trump fails, his team sure doesn t...",News,"March 27, 2017",0
17117,MILITARY LEADERS SPEAK UP: IRAN DEAL MAKES WAR...,John Kerry insinuated that war would be more l...,Government News,"Sep 3, 2015",0
18408,BROKE City of Chicago Spends Taxpayer Money St...,The Windy City is under fire for turning publi...,left-news,"Jun 30, 2017",0
17014,London will remain leading financial center: P...,LONDON (Reuters) - London will remain the worl...,worldnews,"October 19, 2017",1
9360,WHAT A FOURSOME! Trump Plays With Golf Greats ...,President Trump announced he d be playing golf...,politics,"Nov 24, 2017",0
22842,Trump Announces Transgender Ban for US Militar...,"21st Century Wire says On Twitter today, Presi...",Middle-east,"July 26, 2017",0
14687,"HILLARY LANDS COVETED Taxpayer Funded, Planned...","Sadly, this will be the only reason many women...",politics,"Jan 8, 2016",0
17625,"Brexit talks deadlock on cash, Barnier eyes mo...",BRUSSELS (Reuters) - Brexit talks are deadlock...,worldnews,"October 12, 2017",1
13251,ANGRY BLACK MILWAUKEE RESIDENTS Set City On Fi...,This is Obama s America This will be his legac...,politics,"Aug 14, 2016",0
20756,COLLEGE PROFESSOR CAUGHT ON TAPE: You Can’t Ha...,Close your eyes and imagine a white professor ...,left-news,"Apr 9, 2016",0


### 3.3. Dropping Unnecessary Columns
The 'title', 'subject', and 'date' columns are not required for our text-based classification model. We drop them to simplify the dataset. The index is also reset.

In [10]:
data = data.drop(["title", "subject", "date"], axis = 1)

In [11]:
data.reset_index(inplace=True)

In [12]:
data.drop(['index'], axis = 1, inplace=True)

In [13]:
data.sample(5)

,text,class
32324,WASHINGTON (Reuters) - FBI Director James Come...,1
40964,"AIN ISSA, Syria (Reuters) - Remaining Islamic ...",1
40875,BAGHDAD (Reuters) - Iraqi forces have captured...,1
8584,When Sarah Palin blamed President Obama for he...,0
3333,This is despicable and morbid.Donald trump has...,0


### 3.4. Defining the Text Cleaning Function
This function, `cleanText`, will perform several text preprocessing steps: converting text to lowercase, removing punctuation, links, and other non-alphanumeric characters to prepare it for vectorization.

In [14]:
def cleanText(text):
    text = text.lower()
    text = re.sub('\[.*?\]',"",text)
    text = re.sub("\\W", " ", text)
    text = re.sub("https?:://\S+|www\. \S+","",text)
    text = re.sub("<.*?>+","",text)
    text = re.sub("[%s]" % re.escape(string.punctuation),"",text)
    text = re.sub("\n","",text)
    text = re.sub("w*\d\w*","",text)
    return text

### 3.5. Applying the Cleaning Function
The `cleanText` function is applied to the 'text' column of our DataFrame. This step cleans all the news articles in our dataset.

In [15]:
data["text"] = data["text"].apply(cleanText)

## 4. Feature Engineering and Data Splitting

In [16]:
x=data["text"]
y=data["class"]

xtrain, xtest, ytrain, ytest = train_test_split(x,y,test_size=0.25,random_state=42)

### 4.1. Splitting Data into Features (X) and Target (y)
We separate our data into features (the article text, `x`) and the target label (the 'class', `y`). The data is then split into training (75%) and testing (25%) sets.

### 4.2. Vectorizing the Text Data with TF-IDF
Machine learning models require numerical input. We use the `TfidfVectorizer` to convert our text data into numerical vectors. The vectorizer is `fit` on the training data and then used to `transform` both the training and testing data.

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
xv_train = vectorizer.fit_transform(xtrain)
xv_test = vectorizer.transform(xtest)

## 5. Model Training and Comparative Evaluation
Now we will train and evaluate each of our four selected models.

### 5.1. Logistic Regression

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
lr = LogisticRegression()
lr.fit(xv_train,ytrain)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [20]:
prediction = lr.predict(xv_test)
lr.score(xv_test,ytest)

0.9861024498886414

In [21]:
print(classification_report(ytest,prediction))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5895
           1       0.98      0.99      0.99      5330

    accuracy                           0.99     11225
   macro avg       0.99      0.99      0.99     11225
weighted avg       0.99      0.99      0.99     11225



### 5.2. Decision Tree Classifier

In [30]:
from sklearn.tree import DecisionTreeClassifier
DT = DecisionTreeClassifier()
DT.fit(xv_train, ytrain)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [31]:
DecisionTreeClassifier()

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [27]:
pred_dt = DT.predict(xv_test)

In [28]:
DT.score(xv_test, ytest)

0.9953674832962138

In [32]:
print(classification_report(ytest, prediction))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5895
           1       0.98      0.99      0.99      5330

    accuracy                           0.99     11225
   macro avg       0.99      0.99      0.99     11225
weighted avg       0.99      0.99      0.99     11225



### 5.3. Gradient Boosting Classifier

In [33]:
from sklearn.ensemble import GradientBoostingClassifier
GB = GradientBoostingClassifier()
GB.fit(xv_train, ytrain)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [34]:
pred_gb = GB.predict(xv_test)

In [35]:
GB.score(xv_test, ytest)

0.9951002227171493

In [36]:
print(classification_report(ytest, pred_gb))

              precision    recall  f1-score   support

           0       1.00      0.99      1.00      5895
           1       0.99      1.00      0.99      5330

    accuracy                           1.00     11225
   macro avg       0.99      1.00      1.00     11225
weighted avg       1.00      1.00      1.00     11225



### 5.4. Random Forest Classifier

In [37]:
from sklearn.ensemble import RandomForestClassifier
RF = RandomForestClassifier(random_state = 0)
RF.fit(xv_train, ytrain)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [38]:
pred_rf = RF.predict(xv_test)

In [39]:
RF.score(xv_test, ytest)

0.9893095768374165

In [40]:
print(classification_report(ytest, pred_rf))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5895
           1       0.99      0.99      0.99      5330

    accuracy                           0.99     11225
   macro avg       0.99      0.99      0.99     11225
weighted avg       0.99      0.99      0.99     11225



## 6. Saving the Trained Models
The best-performing models are saved to disk using `joblib`. This allows us to load and use them in a separate application (like a web app) without needing to retrain them.

In [64]:
joblib.dump(DT,"DT_model.jb")
joblib.dump(GB, "GB_model.jb")
joblib.dump(RF,"RF_model.jb")

['RF_model.jb']